# Stage 1: ISM Labeling

Matches each EIS `.ism` file to the furnace log windows, classifies it as
VALID / UNSTABLE / OUTSIDE_RANGE, and auto-generates its temperature label.
VALID files are copied to `ISM validation/`.

Raw data are never modified: only copies are made.

**Reads:** `{sample_id}/Raw data/{condition}/*.ism` · `{sample_id}/Raw oven*/*.txt`
**Writes:** `{sample_id}/ISM validation/{condition}/` · `Results/{condition}/stage1_labeling.xlsx`

## Quick links
- [Configuration](#configuration): sample_id, thresholds
- [Import](#import): libraries, sample directory, available conditions
- [Matching + Verification](#matching--verification): VALID/UNSTABLE tables
- [Verification plot](#verification-oven-profile-with-ism-selection): oven profile with ISM selection
- [Export Excel](#export-excel-tables): stage1_labeling.xlsx
- [Copy VALID files](#copy-valid-files-into-ism-validation): copy to ISM validation/ 

## Configuration

**Mode switch** `PARAM_MODE` (top of the cell below), hand-edited and the cell re-run to switch:

- `"lock"` = read-only reproduction of the saved calibration, nothing writes back.
- `"continue"` = config cell is the base, starting values load from `session.json` when present, widget/Apply edits merge-save.
- `"reset"` = deliberately ignore `session.json`, start from the literals below, next save overwrites the saved history.

Full semantics: README, "Changing parameters".

In [ ]:
import json
from pathlib import Path
from pipeline.interactive import select_sample, param_source_banner
from pipeline.session import load_sample, update_sample

NOTEBOOK_DIR = Path.cwd()

sample_id = select_sample(NOTEBOOK_DIR, show_list=True)

_cfg = load_sample(sample_id)

# lock: read-only, no write.
# continue: read session.json if present, edits merge-save.
# reset: ignores session.json, starts from literals below, next save overwrites.
# Full semantics: README, "Changing parameters".
PARAM_MODE = "continue"

_LOCKED_MSG = ("locked (PARAM_MODE='lock'): not saved. "
               "Set PARAM_MODE to 'continue' or 'reset' in Configuration to edit.")

def _update_session(**fields) -> bool:
    """Merge-save to session.json. No-op (returns False) in lock mode."""
    if PARAM_MODE == "lock":
        return False
    update_sample(sample_id, **fields)
    return True

# Starting values load from session.json unless reset: re-run must never
# silently wipe saved tuning.
_p1 = {} if PARAM_MODE == "reset" else _cfg.get("stage1_params", {})
param_source_banner(PARAM_MODE, "Stage 1")
T_STABILITY_STD   = _p1.get("T_STABILITY_STD",   1)
T_PRE_MARGIN_MIN  = _p1.get("T_PRE_MARGIN_MIN",  25)
T_POST_MARGIN_MIN = _p1.get("T_POST_MARGIN_MIN",  5)
T_ROUND_STEP      = _p1.get("T_ROUND_STEP",       25)
T_PLATEAU_RANGE   = tuple(_p1.get("T_PLATEAU_RANGE", [395, 605]))

_ = _update_session(stage1_params={
    "T_STABILITY_STD":   T_STABILITY_STD,
    "T_PRE_MARGIN_MIN":  T_PRE_MARGIN_MIN,
    "T_POST_MARGIN_MIN": T_POST_MARGIN_MIN,
    "T_ROUND_STEP":      T_ROUND_STEP,
    "T_PLATEAU_RANGE":   list(T_PLATEAU_RANGE),
})

## Import

In [ ]:
import sys
import shutil
import json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import display

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.ingest import (
    scan_condition_dir, records_to_dataframe, extract_T_from_filename
)
from pipeline.matching import (
    find_furnace_log, parse_oven_file,
    match_ism_to_furnace, validate_against_filename_labels,
    get_label_prefix, build_auto_labels, validate_auto_labels,
    plot_ism_selection,
)

sample_dir     = NOTEBOOK_DIR / sample_id
raw_data_dir   = sample_dir / "Raw data"
all_conditions = sorted([d.name for d in raw_data_dir.iterdir() if d.is_dir()])
parsed_logs    = {}

_saved     = _cfg.get("conditions", [])
conditions = [c for c in all_conditions if c in _saved] if _saved else all_conditions

print(f"Sample     : {sample_id}")
print(f"Conditions : {len(conditions)}/{len(all_conditions)} from session.json")
for c in conditions:
    print(f"  {c}")

## Matching + Verification

Runs matching for all conditions and prints VALID/UNSTABLE tables.

Before proceeding, verify:
- T_nominal matches expected plateau temperatures
- T_std < T_STABILITY_STD for all VALID files
- pO2_mean is consistent with the gas condition
- UNSTABLE files correspond to ramp / transition periods

In [ ]:
all_results = {}  # condition → list of IsmRecord

# True: display the full per-condition tables (VALID, rejected, mismatches).
# False: one summary row per condition; details appear automatically when
# something needs attention (T mismatch or ordering error).
SHOW_DETAILS = False

pd.set_option("display.float_format", "{:.6f}".format)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 280)

summary_rows = []

for condition_folder in conditions:
    condition_dir = sample_dir / "Raw data" / condition_folder

    try:
        furnace_log_path = find_furnace_log(sample_dir, condition_folder)
    except FileNotFoundError as e:
        print(f"[SKIP] {condition_folder}: no furnace log ({e})")
        continue

    parsed = parse_oven_file(furnace_log_path)
    furnace_df = parsed["df"]
    parsed_logs[condition_folder] = parsed

    records = scan_condition_dir(condition_dir)
    if not records:
        print(f"[SKIP] {condition_folder}: no .ism files; "
              "check the condition folder name and contents.")
        continue

    records = match_ism_to_furnace(
        records, furnace_df,
        T_stability_std  = T_STABILITY_STD,
        T_round_step     = T_ROUND_STEP,
        T_plateau_range  = T_PLATEAU_RANGE,
        pre_margin_min   = T_PRE_MARGIN_MIN,
        post_margin_min  = T_POST_MARGIN_MIN,
    )

    prefix  = get_label_prefix(records, condition_folder)
    records = build_auto_labels(records, prefix)

    status_counts = Counter(r.status for r in records)
    n_labeled = sum(1 for r in records if r.status == "VALID"
                    and extract_T_from_filename(r.path.name) is not None)
    n_seq     = sum(1 for r in records if r.status == "VALID"
                    and extract_T_from_filename(r.path.name) is None)

    per_T = defaultdict(int)
    for r in records:
        if r.status == "VALID" and r.T_nominal is not None:
            per_T[r.T_nominal] += 1

    df_disp   = records_to_dataframe(records)
    val_basic = validate_against_filename_labels(records)
    val_auto  = validate_auto_labels(records)

    df_disp = df_disp.merge(
        val_basic[["file", "T_file_label", "label_match"]], on="file", how="left"
    )
    df_disp["T_file_label"] = df_disp["T_file_label"].fillna("renamed")
    df_disp = df_disp.merge(
        val_auto[["file", "auto_label", "T_match", "order_ok"]], on="file", how="left"
    )

    df_disp["T_ok"] = df_disp["T_match"].map({True: "✓", False: "✗ MISMATCH", None: "auto"})
    df_disp["ord"]  = df_disp["order_ok"].map({True: "✓", False: "✗ ORDER ERR", None: "auto"})

    df_sorted = pd.concat([
        df_disp[df_disp["status"] == "VALID"],
        df_disp[df_disp["status"] != "VALID"],
    ]).reset_index(drop=True)

    rejected       = df_sorted[df_sorted["status"] != "VALID"]
    t_mismatches   = val_auto[val_auto["T_match"]  == False]
    ord_mismatches = val_auto[val_auto["order_ok"] == False]

    summary_rows.append({
        "condition":  condition_folder,
        "log":        furnace_log_path.name,
        "ism_files":  len(records),
        "valid":      status_counts.get("VALID", 0),
        "labeled":    n_labeled,
        "sequential": n_seq,
        "rejected":   sum(v for k, v in status_counts.items() if k != "VALID"),
        "per_T":      "  ".join(f"{int(T)}°Cx{n}"
                                for T, n in sorted(per_T.items(), reverse=True)),
        "T_mismatch": len(t_mismatches),
        "order_err":  len(ord_mismatches),
    })

    if SHOW_DETAILS or len(t_mismatches) or len(ord_mismatches):
        print(f"\n{'='*70}\nCondition: {condition_folder}  (prefix '{prefix}')\n{'='*70}")
        print("── VALID files ──────────────────────────────────────────────────────")
        valid_only = df_sorted[df_sorted["status"] == "VALID"]
        display(valid_only[[
            "file", "t_start", "T_mean", "T_std", "pO2_mean",
            "T_nominal", "replica", "auto_label",
            "T_file_label", "T_ok", "ord", "status",
        ]])
        if len(rejected):
            print(f"── Rejected ({len(rejected)} files) ─────────────────────────────────────")
            display(rejected[["file", "t_start", "T_mean", "T_std", "status"]])
        if len(t_mismatches):
            print(f"  {len(t_mismatches)} T MISMATCH(ES): furnace T != filename T:")
            display(t_mismatches[["file", "T_nominal", "T_file", "auto_label"]])
        if len(ord_mismatches):
            print("  ORDER ERROR: labeled files not in chronological order:")
            display(ord_mismatches[["file", "T_nominal", "replica_seq", "replica_file"]])

    all_results[condition_folder] = records

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    # Hide the check columns when every condition is clean; they reappear on any real mismatch
    if summary_df["T_mismatch"].sum() == 0 and summary_df["order_err"].sum() == 0:
        summary_df = summary_df.drop(columns=["T_mismatch", "order_err"])
        n_checked = int(summary_df["labeled"].sum())
        print(f"T labels and replica order: OK ({n_checked} labeled files checked)")
    print("Matching summary (one row per condition):")
    display(summary_df)

print(f"Matching complete: {len(all_results)} condition(s) ready.")
print("Full per-file tables are exported by the Excel cell below; "
      "set SHOW_DETAILS=True to display them here.")
print("Run the VERIFICATION PLOTS cell below before copying.")

## Verification: oven profile with ISM selection

Shows selected (green), near-transition (orange), and unstable (red) measurements
overlaid on the furnace temperature profile.

If the selection looks wrong, adjust T_STABILITY_STD or T_PRE_MARGIN_MIN in the config cell,
then re-run matching and this cell.

In [ ]:
for condition_folder, records in all_results.items():
    parsed = parsed_logs.get(condition_folder)
    if parsed is None:
        print(f"[SKIP] No furnace log cached for {condition_folder}")
        continue

    results_dir = sample_dir / "Results" / condition_folder / "Oven"
    results_dir.mkdir(parents=True, exist_ok=True)
    save_path = results_dir / f"ism_selection_{condition_folder}.png"

    plot_ism_selection(
        parsed, records,
        condition_folder=condition_folder,
        save_path=save_path,
        show=True,
    )

## Export Excel tables

Saves `stage1_labeling.xlsx` per condition (All + VALID + Metadata sheets), input for Stage 2.

In [ ]:
from pipeline.utils import build_metadata_sheet

df_meta = build_metadata_sheet(
    sample_id  = sample_id,
    stage_name = "stage1_labeling",
    params = {
        "T_STABILITY_STD":   T_STABILITY_STD,
        "T_PRE_MARGIN_MIN":  T_PRE_MARGIN_MIN,
        "T_POST_MARGIN_MIN": T_POST_MARGIN_MIN,
        "T_ROUND_STEP":      T_ROUND_STEP,
        "T_PLATEAU_RANGE":   str(T_PLATEAU_RANGE),
        "validation_rule":   "VALID when T_std < T_STABILITY_STD AND T_mean in T_PLATEAU_RANGE",
        "reference":         "Stage 1 furnace-log matching (pipeline/matching.py)",
    },
)

for condition_folder, records in all_results.items():
    results_dir = sample_dir / "Results" / condition_folder
    results_dir.mkdir(parents=True, exist_ok=True)

    df_exp    = records_to_dataframe(records)
    val_auto  = validate_auto_labels(records)
    df_exp    = df_exp.merge(
        val_auto[["file", "auto_label", "T_match", "order_ok"]], on="file", how="left"
    )
    path_map = {r.path.name: str(r.path) for r in records}
    df_exp["full_path"] = df_exp["file"].map(path_map)

    xlsx_path = results_dir / "stage1_labeling.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df_exp.to_excel(writer, sheet_name="All",   index=False)
        df_exp[df_exp["status"] == "VALID"].reset_index(drop=True).to_excel(
            writer, sheet_name="VALID", index=False)
        df_meta.to_excel(writer, sheet_name="Metadata", index=False)

    n_valid    = (df_exp["status"] == "VALID").sum()
    n_unstable = (df_exp["status"] == "UNSTABLE").sum()
    print(f"  [{condition_folder}]  Total: {len(df_exp)} | VALID: {n_valid} | UNSTABLE: {n_unstable}")
    print(f"    -> {xlsx_path.relative_to(NOTEBOOK_DIR)}")

print("\nExcel export done.")

## Copy VALID files into ISM validation/

Only run after reviewing the verification tables above.

- Raw data/ remains untouched.
- Files are copied, not moved
- Already-labeled files (matching `_{T}C.ism` or `_{T}C_{n}.ism`, for example `_400C.ism` or `_400C_1.ism`) are copied with the original name
- Sequential files are copied with auto-generated labels (`_{T}C.ism` for the first replica, `_{T}C_{n}.ism` for later replicas, per `generate_auto_label()` in `pipeline/matching.py`)
- Files already present are skipped. 

In [ ]:
import re as _re
_LABELED_PAT = _re.compile(r"_\d{2,4}[Cc](?:_\d+)?\.ism$", _re.IGNORECASE)

total_copied = total_skipped = total_renamed = 0

for condition_folder, records in all_results.items():
    validation_dir = sample_dir / "ISM validation" / condition_folder
    validation_dir.mkdir(parents=True, exist_ok=True)

    valid_recs = [r for r in records if r.status == "VALID"]
    copied = skipped = renamed = 0

    for r in valid_recs:
        is_labeled = bool(_LABELED_PAT.search(r.path.name))

        if is_labeled:
            # Already has a temperature label → keep original name
            dst_name = r.path.name
        else:
            # Sequential file → use auto-generated label
            dst_name = r.auto_label or r.path.name   # fallback: keep original if label missing

        dst = validation_dir / dst_name

        if dst.exists():
            skipped += 1
        else:
            shutil.copy2(r.path, dst)
            if not is_labeled:
                print(f"    [RENAME] {r.path.name}  ->  {dst_name}")
                renamed += 1
            else:
                copied += 1

    total_copied  += copied
    total_skipped += skipped
    total_renamed += renamed

    print(f"  [{condition_folder}]")
    print(f"    Copied (labeled)  : {copied}")
    print(f"    Renamed (auto)    : {renamed}")
    print(f"    Skipped (exist)   : {skipped}")

    # Summary per T group
    groups = defaultdict(int)
    for f in sorted(validation_dir.glob("*.ism")):
        T = extract_T_from_filename(f.name)
        groups[T] += 1
    labeled_groups = {T: n for T, n in groups.items() if T is not None}
    unlabeled      = groups.get(None, 0)
    print(f"    Files per T: " + "  ".join(
        f"{T}°Cx{n}" for T, n in sorted(labeled_groups.items(), reverse=True)
    ))
    if unlabeled:
        print(f"    + {unlabeled} still-sequential files (no auto_label assigned)")
    print()

print(f"Total copied (labeled) : {total_copied}")
print(f"Total renamed (auto)   : {total_renamed}")
print(f"Total skipped          : {total_skipped}")

**Next step:** run [stage2_kk.ipynb](stage2_kk.ipynb)